# CH4 — LangChain Agents
## 03 · Agent Memory

### The Problem: Agents are Stateless by Default

Without memory, every call to an agent is a **fresh start**:
```
Turn 1: "My name is Arun"  → Agent: "Hello Arun!"
Turn 2: "What is my name?" → Agent: "I don't know your name" ← PROBLEM!
```

### Solution: Pass Chat History

LangChain agents accept a `messages` list — we can maintain this list across turns:
```
Turn 1: messages = [("user", "My name is Arun")]
Turn 2: messages = [history..., ("user", "What is my name?")]  ← includes past
```

### 3 Approaches
| Approach | Best For |
|----------|----------|
| Manual history | Full control, custom logic |
| `RunnableWithMessageHistory` | Clean API, store-backed |
| In-memory chat history | Multi-user / multi-session |

In [14]:
import os
from load_dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city. Returns temperature and conditions."""
    weather_data = {
        "mumbai": "32°C, Humid",
        "delhi": "38°C, Sunny",
        "bangalore": "24°C, Cloudy",
        "chennai": "34°C, Partly cloudy",
    }
    return weather_data.get(city.lower(), f"Weather data unavailable for {city}")

wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1),
    description="Look up facts on Wikipedia",
)

agent = create_agent(model=llm, tools=[get_weather, wiki_tool])
print("Agent ready")

Agent ready


## Approach 1 — Manual History
Maintain the `messages` list yourself. Simple and explicit.

In [15]:
from langchain_core.messages import HumanMessage, AIMessage

def chat_with_agent(agent, conversation_history: list, user_input: str) -> str:
    """Send a message and update history."""
    conversation_history.append(HumanMessage(content=user_input))

    result = agent.invoke({"messages": conversation_history})
    response = result["messages"][-1]

    # Store full message chain (includes tool calls)
    conversation_history.extend(result["messages"][len(conversation_history):])

    return response.content


# Start a conversation
history = []

print("=" * 60)
print("TURN 1")
print("=" * 60)
reply = chat_with_agent(agent, history, "Hi! My name is Arun and I live in Mumbai.")
print(f"Agent: {reply}\n")

print("=" * 60)
print("TURN 2")
print("=" * 60)
reply = chat_with_agent(agent, history, "What is the weather in my city?")
print(f"Agent: {reply}\n")  # Should know Mumbai from Turn 1

print("=" * 60)
print("TURN 3")
print("=" * 60)
reply = chat_with_agent(agent, history, "What is my name?")
print(f"Agent: {reply}")  # Should remember Arun

TURN 1
Agent: Hello Arun! It's great to meet you. How can I assist you today?

TURN 2
Agent: The weather in Mumbai is currently 32°C and humid. Is there anything else you'd like to know?

TURN 3
Agent: Your name is Arun. How can I assist you further?


## Approach 2 — `RunnableWithMessageHistory`
LangChain's built-in wrapper for message history. Automatically injects stored history keyed by `session_id`.

In [16]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# In-memory store: session_id → chat history
session_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Return (or create) a history object for the given session."""
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]


# Wrap the agent with message history
agent_with_history = RunnableWithMessageHistory(
    agent,
    get_session_history,
    input_messages_key="messages",
)

print("Agent with persistent memory ready")

Agent with persistent memory ready


In [17]:
# Clear old runs so the demo starts fresh when this cell is re-executed
session_store.clear()

# Session config — each user gets their own session_id
session_config = {"configurable": {"session_id": "user_arun"}}

def ask(question: str) -> str:
    result = agent_with_history.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=session_config,
    )
    return result["messages"][-1].content

print("Turn 1:", ask("My name is Arun and I'm interested in AI."))
print()
print("Turn 2:", ask("What topics am I interested in?"))
print()
print("Turn 3:", ask("What is my name?"))

Turn 1: Nice to meet you, Arun! AI is a fascinating field with many applications. What specific aspects of AI are you interested in? For example, are you curious about machine learning, natural language processing, robotics, or something else?

Turn 2: Based on your introduction, it seems you're interested in the field of artificial intelligence (AI). However, if you could provide a bit more information about your specific interests within AI—such as applications, recent advancements, ethical considerations, or specific technologies—I could offer a more tailored response or information on those topics!

Turn 3: Your name is Arun.


## Approach 3 — Multi-Session Memory
Different users get completely isolated conversations using different `session_id` values.

In [18]:
def ask_as_user(user_id: str, question: str) -> str:
    result = agent_with_history.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"session_id": user_id}},
    )
    return result["messages"][-1].content

# Two separate users having separate conversations
print("[User: Alice] ", ask_as_user("alice", "Hi, I'm Alice and I love Python."))
print()
print("[User: Bob]   ", ask_as_user("bob", "Hi, I'm Bob and I love Java."))
print()
print("[User: Alice] ", ask_as_user("alice", "What language do I love?"))  # Python
print()
print("[User: Bob]   ", ask_as_user("bob", "What language do I love?"))   # Java

[User: Alice]  Hi Alice! It's great to hear that you love Python. It's a versatile programming language that's used in many fields. What specific aspects of Python do you enjoy the most? Are you working on any projects?

[User: Bob]    Hi Bob! It's great to hear you love Java. What do you enjoy most about it? Are you working on any specific projects or learning something new in Java?

[User: Alice]  You love Python!

[User: Bob]    You love Java!


In [19]:
# Inspect stored history
print("Sessions in store:", list(session_store.keys()))
print()
for session_id, history in session_store.items():
    print(f"--- {session_id} ({len(history.messages)} messages) ---")
    for msg in history.messages:
        role = "User" if msg.type == "human" else "Agent"
        print(f"  {role}: {msg.content[:80]}")
    print()

Sessions in store: ['user_arun', 'alice', 'bob']

--- user_arun (15 messages) ---
  User: My name is Arun and I'm interested in AI.
  User: My name is Arun and I'm interested in AI.
  Agent: Nice to meet you, Arun! AI is a fascinating field with many applications. What s
  User: What topics am I interested in?
  User: My name is Arun and I'm interested in AI.
  Agent: Nice to meet you, Arun! AI is a fascinating field with many applications. What s
  User: What topics am I interested in?
  Agent: Based on your introduction, it seems you're interested in the field of artificia
  User: What is my name?
  User: My name is Arun and I'm interested in AI.
  Agent: Nice to meet you, Arun! AI is a fascinating field with many applications. What s
  User: What topics am I interested in?
  Agent: Based on your introduction, it seems you're interested in the field of artificia
  User: What is my name?
  Agent: Your name is Arun.

--- alice (8 messages) ---
  User: Hi, I'm Alice and I love Python.
 

## Summary

| Approach | Use When |
|----------|----------|
| Manual history | Simple apps, full control over message list |
| `RunnableWithMessageHistory` | Production apps with multiple sessions |
| Database-backed store | Persistent conversations across restarts |

**Key insight:** The agent itself is stateless. Memory lives in the **message list** passed to it.